# Chapter 7 &mdash; Brzozowski's Minimization: Reverse, Determinize, Twice

**Concept 10 of the Chapter 7 decomposition:** *Brzozowski's Minimization: Reverse, Determinize, Reverse, Determinize*

`nfa2dfa(rev_dfa(nfa2dfa(rev_dfa(D))))` &mdash; minimization with no distinguishability table at all.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Brzozowski-Minimization/Concept-Brzozowski-Minimization.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


**Brzozowski's algorithm** is startling: to minimize a DFA, **reverse it, determinize,
reverse again, determinize again.** That is the whole algorithm.

$$\text{min}(D) = \text{det}(\text{rev}(\text{det}(\text{rev}(D))))$$

No frames, no distinguishability table, no equivalence classes. The reason it works:
determinizing a **reversed** machine automatically produces a machine with no two
equivalent states, because subset-construction states are distinguished by the
*suffixes* they accept.

Jove packages it as `min_dfa_brz`.

## 2. Definitions

### A deliberately bloated DFA

In [ ]:
blimp = md2mc('''DFA
I  : 0 -> A
I  : 1 -> B
A  : 0 -> C
A  : 1 -> D
B  : 0 -> D
B  : 1 -> C
C  : 0 | 1 -> F1
D  : 0 | 1 -> F2
F1 : 0 | 1 -> F1
F2 : 0 | 1 -> F2
''')
print("|Q| =", len(blimp["Q"]))

### The four steps, spelled out

In [ ]:
def brz(D):
    s1 = rev_dfa(D)          # NFA
    s2 = nfa2dfa(s1)         # DFA
    s3 = rev_dfa(s2)         # NFA
    s4 = nfa2dfa(s3)         # DFA -- and it is minimal
    return s1, s2, s3, s4

<!-- nav-strip -->

---

&larr;&nbsp;[Ch7&nbsp;9.&nbsp;Theorem: $L$ is Regular iff Some NFA Recognizes It](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Regular-Iff-NFA/Concept-Regular-Iff-NFA.ipynb) &nbsp;&middot;&nbsp; [**Chapter 7** index](https://github.com/ganeshutah/Jove/blob/master/Chapter7-NFA/README.md) &nbsp;&middot;&nbsp; [Ch7&nbsp;11.&nbsp;Reversal of a DFA Yields an NFA](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Reversal-Yields-NFA/Concept-Reversal-Yields-NFA.ipynb)&nbsp;&rarr;

---

## 3. Tests

Watch the size at each step.

In [ ]:
s1, s2, s3, s4 = brz(blimp)
print("original          : %2d states" % len(blimp["Q"]))
print("1. rev  (NFA)     : %2d states" % len(s1["Q"]))
print("2. det            : %2d states" % len(s2["Q"]))
print("3. rev  (NFA)     : %2d states" % len(s3["Q"]))
print("4. det  (minimal) : %2d states" % len(s4["Q"]))

The result really is minimal &mdash; same size as `min_dfa`.

In [ ]:
m = min_dfa(blimp)
print("min_dfa      : %d states" % len(m["Q"]))
print("Brzozowski   : %d states" % len(s4["Q"]))
assert len(s4["Q"]) == len(m["Q"])
assert iso_dfa(s4, m)
print("isomorphic to min_dfa's answer? ", iso_dfa(s4, m))

And the language is untouched.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
assert all(accepts_dfa(s4, s) == accepts_dfa(blimp, s) for s in strs)
print("same language on all %d strings up to length 10" % len(strs))

Jove's `min_dfa_brz` is the same four steps in one call.

In [ ]:
b = min_dfa_brz(blimp)
print("min_dfa_brz : %d states, isomorphic to min_dfa: %s"
      % (len(b["Q"]), iso_dfa(b, m)))
assert iso_dfa(b, m)

Doing only **one** reverse-determinize is not enough.

In [ ]:
half = nfa2dfa(rev_dfa(blimp))
print("after one round : %d states (minimal would be %d)" % (len(half["Q"]), len(m["Q"])))
print("the language is the REVERSE at that point, so it cannot be the answer.")

## 4. Animation

The minimal machine Brzozowski's algorithm produces.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa_brz(blimp), FuseEdges=True)

## 5. Exercises


1. Run the four steps on a machine that is already minimal. What happens?
2. Why does determinizing a reversed DFA remove equivalent states automatically?
3. Compare the cost of Brzozowski with the frame algorithm on a 10-state DFA.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 254 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter7-NFA/Concept-Brzozowski-Minimization')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')